IMPORTS

In [1]:
import os, time, json, copy, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split
# Slúži na potláčanie zbytočných chýb
warnings.filterwarnings("ignore")
# Test či načítalo CUDU
print(f"PyTorch: {torch.__version__}")
print(f"CUDA dostupná: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


PyTorch: 2.6.0+cu124
CUDA dostupná: True
GPU: NVIDIA GeForce RTX 4060
VRAM: 8.6 GB


KONFIGURÁCIA

In [2]:
# Cesty
DATA_DIR    = "./data"
RESULTS_DIR = "./results"

#device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Hyperparametre
INPUT_SIZE   = 224     # VGG16 vstup
BATCH_SIZE   = 64      #menšia batchsize kvôli pretekaniu z VRAM do Shared Memory
NUM_EPOCHS   = 10
NUM_CLASSES  = 10
VAL_SPLIT    = 0.1     # 10 % z train = validácia
SEED         = 42

# Learning rates
LR_HEAD      = 1e-3    # klasifikácia
LR_BACKBONE  = 1e-4    # konvolučné vrstvy
WEIGHT_DECAY = 1e-4

# Triedy CIFAR-10
CIFAR10_CLASSES = ["airplane", "automobile", "bird", "cat", "deer",
                   "dog", "frog", "horse", "ship", "truck"]

torch.manual_seed(SEED)
np.random.seed(SEED)


DATASET

In [3]:
# ImageNet normalizácia - RGB normalizujeme
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# Transformácie
train_transform = transforms.Compose([
    transforms.Resize((INPUT_SIZE, INPUT_SIZE)),
    # Augmentácia
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomCrop(INPUT_SIZE, padding=8),
    transforms.ColorJitter(brightness=0.2, contrast=0.2,
                           saturation=0.2, hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

val_test_transform = transforms.Compose([
    transforms.Resize((INPUT_SIZE, INPUT_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

no_aug_transform = transforms.Compose([
    transforms.Resize((INPUT_SIZE, INPUT_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


In [4]:
def get_dataloaders(batch_size=BATCH_SIZE, val_split=VAL_SPLIT,
                    use_augmentation=True):
    # Načíta CIFAR-10 a rozdelí na train / val / test
    t_transform = train_transform if use_augmentation else no_aug_transform

    full_train = datasets.CIFAR10(root=DATA_DIR, train=True,
                                  download=True, transform=t_transform)
    test_set   = datasets.CIFAR10(root=DATA_DIR, train=False,
                                  download=True, transform=val_test_transform)

    n_val   = int(len(full_train) * val_split)
    n_train = len(full_train) - n_val
    train_set, val_set = random_split(
        full_train, [n_train, n_val],
        generator=torch.Generator().manual_seed(SEED)
    )
    # Validačná sada bez augmentácie
    val_ds = copy.deepcopy(full_train)
    val_ds.transform = val_test_transform
    val_set.dataset = val_ds
    # num_workers a pin_memory -----> CUDA
    loaders = {
        "train": DataLoader(train_set, batch_size=batch_size,
                            shuffle=True,  num_workers=4, pin_memory=True), 
        "val":   DataLoader(val_set,   batch_size=batch_size,
                            shuffle=False, num_workers=4, pin_memory=True),
        "test":  DataLoader(test_set,  batch_size=batch_size,
                            shuffle=False, num_workers=4, pin_memory=True),
    }
    print(f"Train: {n_train:,} | Val: {n_val:,} | Test: {len(test_set):,}")
    return loaders

# Načítaj dáta
loaders        = get_dataloaders(use_augmentation=True)
loaders_no_aug = get_dataloaders(use_augmentation=False)


Train: 45,000 | Val: 5,000 | Test: 10,000
Train: 45,000 | Val: 5,000 | Test: 10,000


Build VGG-16 REDEP

In [5]:
def build_vgg16(num_classes=NUM_CLASSES, dropout=0.5):
    #Predtrénovaný VGG16 s novou klasifikačnou hlavou
    model = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)
    in_features = model.classifier[6].in_features  # 4096
    model.classifier[6] = nn.Sequential(
        nn.Dropout(p=dropout),
        nn.Linear(in_features, num_classes),
    )
    return model

# Ukáž architektúru
model_preview = build_vgg16()
print(model_preview)

# Info o parametroch
total = sum(p.numel() for p in model_preview.parameters())
print(f"Celkový počet parametrov VGG16: {total:,}")
del model_preview


VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1

Funkcia na nastavenie zmrazených vrstiev

In [6]:
#index prvej vrstvy kazdeho bloku v model.features
BLOCK_START = {1: 0, 2: 5, 3: 10, 4: 20, 5: 30}

def set_frozen_layers(model, unfreeze_from_block=None):
    """
    unfreeze_from_block ked je:
        None   → len head       (E1 – frozen feature extraction)
        5      → block5 + head  (E2)
        4      → block4-5 +head (E3)
        3      → block3-5 + head(E4)
      'all'  → cela siet       (E5 – full fine-tuning)
    """
    for param in model.parameters():
        param.requires_grad = False
    for param in model.classifier.parameters():
        param.requires_grad = True

    if unfreeze_from_block == "all":
        for param in model.parameters():
            param.requires_grad = True
    elif unfreeze_from_block is not None:        #ak je zadane cislo bloku
        start = BLOCK_START[unfreeze_from_block]
        for i, layer in enumerate(model.features):
            if i >= start:
                for param in layer.parameters():
                    param.requires_grad = True

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    print(f"  Trénovateľné: {trainable:,} / {total:,}  "
          f"({100*trainable/total:.1f} %)")
    return model


def get_optimizer(model):
    #AdamW s oddelenými LR pre hlavu a backbone
    head_ids        = set(id(p) for p in model.classifier.parameters())
    backbone_params = [p for p in model.parameters()
                       if p.requires_grad and id(p) not in head_ids]
    groups = [{"params": list(model.classifier.parameters()), "lr": LR_HEAD}] #skupiny s roznymi learning rates
    if backbone_params:
        groups.append({"params": backbone_params, "lr": LR_BACKBONE})
    return optim.AdamW(groups, weight_decay=WEIGHT_DECAY)




Tréningové funkcie

In [7]:
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    loss_sum, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        out  = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        loss_sum += loss.item() * imgs.size(0)
        correct  += out.argmax(1).eq(labels).sum().item()
        total    += imgs.size(0)
    return loss_sum / total, 100.0 * correct / total


@torch.no_grad()
def eval_epoch(model, loader, criterion, device):
    model.eval()
    loss_sum, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        out  = model(imgs)
        loss = criterion(out, labels)
        loss_sum += loss.item() * imgs.size(0)
        correct  += out.argmax(1).eq(labels).sum().item()
        total    += imgs.size(0)
    return loss_sum / total, 100.0 * correct / total


In [8]:
def run_experiment(name, unfreeze_from_block, loaders,
                   epochs=NUM_EPOCHS, device=DEVICE):
    # Spustí jeden experiment, uloží históriu a best model.
    print(f"\n{'='*60}")
    print(f"  EXPERIMENT: {name}")
    print(f"{'='*60}")

    model     = build_vgg16().to(device)
    model     = set_frozen_layers(model, unfreeze_from_block)
    optimizer = get_optimizer(model)
    scheduler = CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
    criterion = nn.CrossEntropyLoss()

    history = {"name": name,
               "train_loss": [], "train_acc": [],
               "val_loss":   [], "val_acc":   [],
               "epoch_time": []}
    best_val_acc, best_weights = 0.0, None
    t_total = time.time()

    for ep in range(1, epochs + 1):
        t0 = time.time()
        tl, ta = train_epoch(model, loaders["train"], criterion, optimizer, device)
        vl, va = eval_epoch(model,  loaders["val"],   criterion, device)
        scheduler.step()
        dt = time.time() - t0

        history["train_loss"].append(round(tl, 4))
        history["train_acc"].append(round(ta, 2))
        history["val_loss"].append(round(vl, 4))
        history["val_acc"].append(round(va, 2))
        history["epoch_time"].append(round(dt, 1))

        if va > best_val_acc:
            best_val_acc  = va
            best_weights  = copy.deepcopy(model.state_dict())

        print(f"  Ep {ep:3d}/{epochs}  |  "
              f"Train  loss:{tl:.4f}  acc:{ta:.2f}%  |  "
              f"Val  loss:{vl:.4f}  acc:{va:.2f}%  |  {dt:.1f}s")

    history["total_time_min"] = round((time.time() - t_total) / 60, 2)
    history["best_val_acc"]   = round(best_val_acc, 2)

    # Test
    model.load_state_dict(best_weights)
    test_loss, test_acc = eval_epoch(model, loaders["test"], criterion, device)
    history["test_acc"]  = round(test_acc, 2)
    history["test_loss"] = round(test_loss, 4)

    print(f"\n  Best Val: {best_val_acc:.2f}%  |  "
          f"Test: {test_acc:.2f}%  |  "
          f"Čas: {history['total_time_min']} min")

    # Uloženie
    safe = name.replace(" ", "_").replace("/", "-")
    torch.save(best_weights, f"{RESULTS_DIR}/{safe}_best.pth")
    with open(f"{RESULTS_DIR}/{safe}_history.json", "w") as f:
        json.dump(history, f, indent=2)

    return history, model


### E1 – Frozen feature extraction
*len klasifikačná hlava zmrazená*

In [9]:
hist_e1, model_e1 = run_experiment(
    name="E1 – Frozen feature extraction",
    unfreeze_from_block=None,
    loaders=loaders,
    epochs=NUM_EPOCHS,
)


  EXPERIMENT: E1 – Frozen feature extraction
  Trénovateľné: 119,586,826 / 134,301,514  (89.0 %)
  Ep   1/10  |  Train  loss:1.0099  acc:70.17%  |  Val  loss:0.5178  acc:83.30%  |  221.6s
  Ep   2/10  |  Train  loss:0.8709  acc:75.50%  |  Val  loss:0.4682  acc:85.80%  |  221.5s
  Ep   3/10  |  Train  loss:0.7598  acc:78.32%  |  Val  loss:0.4629  acc:85.84%  |  222.8s
  Ep   4/10  |  Train  loss:0.6616  acc:80.68%  |  Val  loss:0.4334  acc:86.38%  |  220.1s
  Ep   5/10  |  Train  loss:0.5927  acc:82.44%  |  Val  loss:0.4108  acc:86.82%  |  219.7s
  Ep   6/10  |  Train  loss:0.5200  acc:84.19%  |  Val  loss:0.3846  acc:87.98%  |  219.5s
  Ep   7/10  |  Train  loss:0.4572  acc:86.16%  |  Val  loss:0.3596  acc:88.20%  |  220.2s
  Ep   8/10  |  Train  loss:0.4139  acc:86.95%  |  Val  loss:0.3522  acc:88.40%  |  220.1s
  Ep   9/10  |  Train  loss:0.3813  acc:88.08%  |  Val  loss:0.3457  acc:88.72%  |  222.3s
  Ep  10/10  |  Train  loss:0.3584  acc:88.58%  |  Val  loss:0.3440  acc:88.60%  | 

### E2 – Unfreeze Block 5
*rozmrazený block 5 + hlava*

In [ ]:
hist_e2, model_e2 = run_experiment(
    name="E2 – Unfreeze Block 5",
    unfreeze_from_block=5,
    loaders=loaders,
    epochs=NUM_EPOCHS,
)



  EXPERIMENT: E2 – Unfreeze Block 5
  Trénovateľné: 119,586,826 / 134,301,514  (89.0 %)
  Ep   1/10  |  Train  loss:0.8207  acc:73.54%  |  Val  loss:0.5118  acc:83.30%  |  689.8s
  Ep   2/10  |  Train  loss:0.7130  acc:78.49%  |  Val  loss:0.4471  acc:85.32%  |  1088.4s
  Ep   3/10  |  Train  loss:0.6356  acc:80.47%  |  Val  loss:0.4082  acc:86.68%  |  1088.5s
  Ep   4/10  |  Train  loss:0.5565  acc:82.82%  |  Val  loss:0.3884  acc:87.64%  |  1088.5s
  Ep   5/10  |  Train  loss:0.4876  acc:84.53%  |  Val  loss:0.3789  acc:87.92%  |  1088.4s
  Ep   6/10  |  Train  loss:0.4251  acc:86.36%  |  Val  loss:0.3615  acc:88.20%  |  1088.3s
  Ep   7/10  |  Train  loss:0.3721  acc:87.95%  |  Val  loss:0.3509  acc:88.32%  |  1088.3s
  Ep   8/10  |  Train  loss:0.3384  acc:88.87%  |  Val  loss:0.3353  acc:88.38%  |  1090.5s
  Ep   9/10  |  Train  loss:0.3104  acc:89.76%  |  Val  loss:0.3281  acc:89.00%  |  1088.4s
  Ep  10/10  |  Train  loss:0.2979  acc:90.15%  |  Val  loss:0.3264  acc:89.08%  |  

### E3 – Unfreeze Block 4–5
*rozmrazené bloky 4–5 + hlava*

In [ ]:
hist_e3, model_e3 = run_experiment(
    name="E3 – Unfreeze Block 4–5",
    unfreeze_from_block=4,
    loaders=loaders,
    epochs=NUM_EPOCHS,
)



  EXPERIMENT: E3 – Unfreeze Block 4–5
  Trénovateľné: 129,026,058 / 134,301,514  (96.1 %)
  Ep   1/10  |  Train  loss:0.5976  acc:80.54%  |  Val  loss:0.3960  acc:86.88%  |  1123.9s
  Ep   2/10  |  Train  loss:0.3708  acc:88.04%  |  Val  loss:0.3099  acc:90.10%  |  1560.8s
  Ep   3/10  |  Train  loss:0.2847  acc:91.01%  |  Val  loss:0.3092  acc:90.52%  |  1732.6s
  Ep   4/10  |  Train  loss:0.2084  acc:93.44%  |  Val  loss:0.3104  acc:90.86%  |  1560.8s
  Ep   5/10  |  Train  loss:0.1625  acc:94.93%  |  Val  loss:0.2956  acc:91.70%  |  1732.8s
  Ep   6/10  |  Train  loss:0.1094  acc:96.56%  |  Val  loss:0.3073  acc:91.64%  |  1560.9s
  Ep   7/10  |  Train  loss:0.0792  acc:97.49%  |  Val  loss:0.3111  acc:92.44%  |  1560.7s
  Ep   8/10  |  Train  loss:0.0535  acc:98.38%  |  Val  loss:0.2993  acc:92.82%  |  1735.0s
  Ep   9/10  |  Train  loss:0.0385  acc:98.77%  |  Val  loss:0.3148  acc:92.94%  |  1152.9s
  Ep  10/10  |  Train  loss:0.0313  acc:98.99%  |  Val  loss:0.3159  acc:93.08%  

### E4 – Unfreeze Block 3–5
*rozmrazené bloky 3–5 + hlava*

In [10]:
hist_e4, model_e4 = run_experiment(
    name="E4 – Unfreeze Block 3–5",
    unfreeze_from_block=3,
    loaders=loaders,
    epochs=NUM_EPOCHS,
)



  EXPERIMENT: E4 – Unfreeze Block 3–5
  Trénovateľné: 134,041,354 / 134,301,514  (99.8 %)
  Ep   1/10  |  Train  loss:0.6447  acc:79.28%  |  Val  loss:0.5330  acc:83.48%  |  403.5s
  Ep   2/10  |  Train  loss:0.4290  acc:86.64%  |  Val  loss:0.3546  acc:88.08%  |  402.9s
  Ep   3/10  |  Train  loss:0.3285  acc:89.83%  |  Val  loss:0.3445  acc:89.58%  |  402.8s
  Ep   4/10  |  Train  loss:0.2566  acc:91.85%  |  Val  loss:0.2951  acc:91.10%  |  402.8s
  Ep   5/10  |  Train  loss:0.1932  acc:93.96%  |  Val  loss:0.2736  acc:91.38%  |  402.7s
  Ep   6/10  |  Train  loss:0.1374  acc:95.59%  |  Val  loss:0.2347  acc:93.04%  |  402.9s
  Ep   7/10  |  Train  loss:0.0867  acc:97.28%  |  Val  loss:0.2449  acc:93.12%  |  402.7s
  Ep   8/10  |  Train  loss:0.0561  acc:98.20%  |  Val  loss:0.2384  acc:93.60%  |  402.8s
  Ep   9/10  |  Train  loss:0.0354  acc:98.92%  |  Val  loss:0.2600  acc:93.88%  |  402.6s
  Ep  10/10  |  Train  loss:0.0227  acc:99.27%  |  Val  loss:0.2593  acc:94.22%  |  392.4s

### E5 – Full fine-tuning
*celá sieť trénovateľná*

In [11]:
hist_e5, model_e5 = run_experiment(
    name="E5 – Full fine-tuning",
    unfreeze_from_block='all',
    loaders=loaders,
    epochs=NUM_EPOCHS,
)


  EXPERIMENT: E5 – Full fine-tuning
  Trénovateľné: 134,301,514 / 134,301,514  (100.0 %)
  Ep   1/10  |  Train  loss:0.6554  acc:78.67%  |  Val  loss:0.4309  acc:86.26%  |  3746.5s
  Ep   2/10  |  Train  loss:0.4316  acc:86.31%  |  Val  loss:0.3793  acc:87.26%  |  5119.6s
  Ep   3/10  |  Train  loss:0.3298  acc:89.47%  |  Val  loss:0.3685  acc:88.80%  |  5991.9s
  Ep   4/10  |  Train  loss:0.2586  acc:91.79%  |  Val  loss:0.3070  acc:90.68%  |  5117.2s
  Ep   5/10  |  Train  loss:0.1969  acc:93.81%  |  Val  loss:0.2777  acc:91.64%  |  5921.5s
  Ep   6/10  |  Train  loss:0.1294  acc:95.81%  |  Val  loss:0.2422  acc:92.86%  |  5685.6s
  Ep   7/10  |  Train  loss:0.0862  acc:97.20%  |  Val  loss:0.2313  acc:93.54%  |  5587.0s
  Ep   8/10  |  Train  loss:0.0520  acc:98.28%  |  Val  loss:0.2436  acc:94.14%  |  5436.2s
  Ep   9/10  |  Train  loss:0.0322  acc:98.95%  |  Val  loss:0.2360  acc:94.00%  |  5716.0s
  Ep  10/10  |  Train  loss:0.0234  acc:99.30%  |  Val  loss:0.2274  acc:94.32%  |

## 7. Ablation Study

**A1 – bez dátovej augmentácie**  
Rovnaká stratégia ako E2 (unfreeze block 5), ale trénovanie prebieha bez
RandomHorizontalFlip, RandomCrop a ColorJitter.  
Cieľ: izolovať vplyv augmentácie na generalizáciu.


In [ ]:
hist_a1, _ = run_experiment(
    name="A1 – no augmentation",
    unfreeze_from_block=5,
    loaders=loaders_no_aug,
    epochs=NUM_EPOCHS,
)


  EXPERIMENT: A1 – no augmentation
  Trénovateľné: 119,586,826 / 134,301,514  (89.0 %)
  Ep   1/15  |  Train  loss:0.9194  acc:73.87%  |  Val  loss:0.5134  acc:83.44%  |  224.4s
  Ep   2/15  |  Train  loss:0.7383  acc:81.29%  |  Val  loss:0.4756  acc:85.60%  |  223.2s
  Ep   3/15  |  Train  loss:0.5970  acc:84.61%  |  Val  loss:0.4849  acc:85.60%  |  222.9s
  Ep   4/15  |  Train  loss:0.5215  acc:86.91%  |  Val  loss:0.4556  acc:87.00%  |  223.6s
  Ep   5/15  |  Train  loss:0.4508  acc:88.84%  |  Val  loss:0.4376  acc:87.70%  |  224.2s
  Ep   6/15  |  Train  loss:0.3655  acc:90.83%  |  Val  loss:0.4381  acc:87.90%  |  223.0s
  Ep   7/15  |  Train  loss:0.3035  acc:92.34%  |  Val  loss:0.4338  acc:87.52%  |  221.5s
  Ep   8/15  |  Train  loss:0.2452  acc:93.86%  |  Val  loss:0.4409  acc:88.34%  |  221.6s
  Ep   9/15  |  Train  loss:0.1994  acc:94.92%  |  Val  loss:0.4394  acc:88.56%  |  223.1s
  Ep  10/15  |  Train  loss:0.1585  acc:96.00%  |  Val  loss:0.4416  acc:89.16%  |  225.7s
  

NameError: name 'all_histories' is not defined

## 8. Vizualizácia výsledkov

In [ ]:
#load all_histories z results (kvôli reštartovaniu kernelu)
def load_history(filename):
    with open(os.path.join(RESULTS_DIR, filename), "r") as f:
        return json.load(f)

# Načítanie jednotlivých histórií
hist_a1 = load_history("A1_–_no_augmentation_history.json")
hist_e1 = load_history("E1_-_Frozen_feature_extraction_history.json")
hist_e2 = load_history("E2_-_Unfreeze_Block_5_history.json")
hist_e3 = load_history("E3_-_Unfreeze_Block_4-5_history.json")
hist_e4 = load_history("E4_-_Unfreeze_Block_3-5_history.json")
hist_e5 = load_history("E5_-_Full_fine-tuning_history.json")

all_histories = [hist_a1, hist_e1, hist_e2, hist_e3, hist_e4, hist_e5]

# Validačná accuracy – všetky stratégie
fig, ax = plt.subplots(figsize=(11, 5))
colors = ["#1f77b4","#ff7f0e","#2ca02c","#d62728","#9467bd","#8c564b"]

for i, h in enumerate(all_histories):
    ax.plot(range(1, len(h["val_acc"]) + 1), h["val_acc"],
            label=h["name"], color=colors[i % len(colors)], linewidth=2)

ax.set_xlabel("Epocha", fontsize=12)
ax.set_ylabel("Validačná accuracy (%)", fontsize=12)
ax.set_title("Porovnanie validačnej accuracy – všetky stratégie", fontsize=13)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/accuracy_val_all.png", dpi=150)
plt.show()


FileNotFoundError: [Errno 2] No such file or directory: './results\\A1_-_no_augmentation_history.json'

In [13]:
# Train vs Val loss – každý experiment zvlášť
main_exps = [hist_e1, hist_e2, hist_e3, hist_e4, hist_e5]
fig, axes = plt.subplots(1, len(main_exps), figsize=(18, 4), sharey=False)

for ax, h in zip(axes, main_exps):
    eps = range(1, len(h["train_loss"]) + 1)
    ax.plot(eps, h["train_loss"], label="Train", linewidth=2)
    ax.plot(eps, h["val_loss"],   label="Val",   linewidth=2, linestyle="--")
    ax.set_title(h["name"], fontsize=8)
    ax.set_xlabel("Epocha")
    ax.set_ylabel("Loss")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle("Priebeh tréningovej a validačnej loss", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/loss_all.png", dpi=150, bbox_inches="tight")
plt.show()


NameError: name 'hist_e2' is not defined

In [ ]:
# Speed–Accuracy scatter
fig, ax = plt.subplots(figsize=(9, 5))
colors2 = ["#1f77b4","#ff7f0e","#2ca02c","#d62728","#9467bd"]

for i, h in enumerate(main_exps):
    ax.scatter(h["total_time_min"], h["test_acc"],
               color=colors2[i], s=160, zorder=5)
    ax.annotate(h["name"],
                (h["total_time_min"], h["test_acc"]),
                textcoords="offset points", xytext=(8, 4), fontsize=9)

ax.set_xlabel("Celkový čas trénovania (min)", fontsize=12)
ax.set_ylabel("Test accuracy (%)", fontsize=12)
ax.set_title("Speed–Accuracy tradeoff", fontsize=13)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/speed_accuracy.png", dpi=150)
plt.show()


In [ ]:
# Ablation: porovnanie E2 vs A1
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Val accuracy
for h, ls in [(hist_e2, "-"), (hist_a1, "--")]:
    axes[0].plot(range(1, len(h["val_acc"]) + 1), h["val_acc"],
                 label=h["name"], linestyle=ls, linewidth=2)
axes[0].set_title("Ablation – Val accuracy")
axes[0].set_xlabel("Epocha"); axes[0].set_ylabel("Val accuracy (%)")
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# Val loss
for h, ls in [(hist_e2, "-"), (hist_a1, "--")]:
    axes[1].plot(range(1, len(h["val_loss"]) + 1), h["val_loss"],
                 label=h["name"], linestyle=ls, linewidth=2)
axes[1].set_title("Ablation – Val loss")
axes[1].set_xlabel("Epocha"); axes[1].set_ylabel("Val loss")
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle("E2 (s augmentáciou) vs A1 (bez augmentácie)", fontsize=13)
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/ablation_e2_vs_a1.png", dpi=150)
plt.show()


## 9. Konfúzna matica a analýza chýb

In [ ]:
@torch.no_grad()
def compute_confusion_matrix(model, loader, n_cls=NUM_CLASSES, device=DEVICE):
    model.eval()
    cm = torch.zeros(n_cls, n_cls, dtype=torch.long)
    for imgs, labels in loader:
        imgs   = imgs.to(device)
        preds  = model(imgs).argmax(1).cpu()
        for t, p in zip(labels, preds):
            cm[t][p] += 1
    return cm.numpy()


In [ ]:
# Vyber najlepší model podľa test accuracy
best_h = max(main_exps, key=lambda h: h.get("test_acc", 0))
print(f"Najlepší experiment: {best_h['name']}  (test acc: {best_h['test_acc']} %)")

# Načítaj váhy
safe_name  = best_h["name"].replace(" ", "_").replace("/", "-")
best_model = build_vgg16().to(DEVICE)
best_model.load_state_dict(
    torch.load(f"{RESULTS_DIR}/{safe_name}_best.pth", map_location=DEVICE)
)

# Vypočítaj konfúznu maticu
cm = compute_confusion_matrix(best_model, loaders["test"])
np.save(f"{RESULTS_DIR}/confusion_matrix_best.npy", cm)
print("Konfúzna matica:")


In [ ]:
# Vykresli konfúznu maticu
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=CIFAR10_CLASSES,
            yticklabels=CIFAR10_CLASSES,
            ax=ax, cbar_kws={"shrink": 0.8})
ax.set_xlabel("Predikovaná trieda", fontsize=12)
ax.set_ylabel("Skutočná trieda", fontsize=12)
ax.set_title(f"Konfúzna matica – {best_h['name']}", fontsize=13)
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/confusion_matrix.png", dpi=150)
plt.show()


In [ ]:
# Top zamenané dvojice tried
confusions = []
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        if i != j:
            confusions.append((cm[i][j], CIFAR10_CLASSES[i], CIFAR10_CLASSES[j]))
confusions.sort(reverse=True)

print("Top 10 najčastejšie zamenané dvojice tried:")
print(f"{'Skutočná':15s} {'Predikovaná':15s} {'Počet chýb':>12s}")
print("-" * 45)
for count, true_cls, pred_cls in confusions[:10]:
    print(f"{true_cls:15s} {pred_cls:15s} {count:>12d}")


In [ ]:
# Per-class accuracy
per_class_acc = cm.diagonal() / cm.sum(axis=1) * 100

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(CIFAR10_CLASSES, per_class_acc,
              color=plt.cm.RdYlGn(per_class_acc / 100))
ax.axhline(per_class_acc.mean(), color="navy", linestyle="--",
           label=f"Priemer: {per_class_acc.mean():.1f}%")
ax.set_ylabel("Accuracy (%)")
ax.set_title("Per-class accuracy – najlepší model")
ax.set_ylim(0, 105)
ax.legend()
for bar, acc in zip(bars, per_class_acc):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f"{acc:.1f}%", ha="center", va="bottom", fontsize=8)
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/per_class_accuracy.png", dpi=150)
plt.show()


## 10. Súhrnná tabuľka výsledkov

In [ ]:
rows = []
for h in all_histories:
    avg_ep = (sum(h.get("epoch_time", [0])) /
              max(len(h.get("epoch_time", [1])), 1))
    rows.append({
        "Stratégia":            h["name"],
        "Best Val Acc (%)":     h.get("best_val_acc", h["val_acc"][-1]),
        "Test Acc (%)":         h.get("test_acc", "–"),
        "Test Loss":            h.get("test_loss", "–"),
        "Čas celkom (min)":     h.get("total_time_min", "–"),
        "Čas/epocha (s)":       round(avg_ep, 1),
        "Epoch":                len(h["val_acc"]),
    })

df_results = pd.DataFrame(rows)
df_results.to_csv(f"{RESULTS_DIR}/summary_results.csv", index=False)

display(df_results.style
        .highlight_max(subset=["Best Val Acc (%)", "Test Acc (%)"],
                       color="lightgreen")
        .highlight_min(subset=["Čas celkom (min)"], color="lightyellow")
        .format(precision=2))

print(f"\nSúhrn uložený do: {RESULTS_DIR}/summary_results.csv")


In [ ]:
# Počet trénovateľných parametrov pre každú stratégiu
print("Počty trénovateľných parametrov:\n")
configs = [
    ("E1 – Frozen", None),
    ("E2 – Block 5", 5),
    ("E3 – Block 4-5", 4),
    ("E4 – Block 3-5", 3),
    ("E5 – Full", "all"),
]
for label, ufb in configs:
    m = build_vgg16()
    set_frozen_layers(m, ufb)
    trainable = sum(p.numel() for p in m.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in m.parameters())
    print(f"  {label:22s}: {trainable:>10,} / {total:>10,}  ({100*trainable/total:.1f}%)")
    del m
